# Causal Finetuning Demo (src-driven)

This notebook is intentionally **src-first**:
- All runtime logic is imported from `src/` modules
- No notebook-local reimplementation of engines, samplers, or orchestrators
- Commands should be executed in the UV-managed environment

In [1]:
# Environment detection (Colab vs local) and workspace setup
from pathlib import Path
import os

IS_COLAB = False
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    WORKSPACE_ROOT = Path('/content/drive/MyDrive/fineTunning').resolve()
else:
    WORKSPACE_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

assert (WORKSPACE_ROOT / 'src').exists(), f'Could not find src/ from {WORKSPACE_ROOT}'
print(f'Workspace root: {WORKSPACE_ROOT}')
print(f'Running on Colab: {IS_COLAB}')

Workspace root: C:\tutoriales\fineTunning
Running on Colab: False


In [2]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
import sys
sys.path.insert(0, str(WORKSPACE_ROOT))

from src.settings import SettingsFactory, CausalTrainingConfig
from src.finetuner.data_loader import GLUEDataLoader
from src.finetuner.lora_engine import FineTuningEngine
from src.finetuner.laplace_engine import LaplaceLoRAEngine
from src.finetuner.causal_engine import CausalMonteCLoRAEngine
from src.utils.metrics import UnifiedEvaluator

print("Core imports loaded from src/.")
print("Using settings package split:", SettingsFactory.__module__)

Core imports loaded from src/.
Using settings package split: src.settings.factory


In [3]:
# Device detection and runtime capabilities
import os
import torch

# Detect device (CUDA if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

available_vram_gb = None
if device == "cuda" and torch.cuda.is_available():
    available_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Default quantization from environment or default
default_quantization = os.environ.get("DEFAULT_QUANTIZATION", "auto")

print("Runtime device:", device)
print("Available VRAM (GB):", available_vram_gb)
print("Default quantization:", default_quantization)

Runtime device: cpu
Available VRAM (GB): None
Default quantization: auto


## Rule Enforcement

When extending this notebook:
1. Import behavior from `src` modules only
2. Do not copy engine logic into notebook cells
3. Keep orchestration and sampling logic in `src/finetuner` and `src/utils`

## main.py-Compatible Multi-Engine Benchmark

The next cells configure separate environment profiles and run a main.py-compatible flow for:
- LoRA Baseline
- Laplace-LoRA
- Causal LoRA

Each run returns a shared metrics row so results can be compared side by side.

In [4]:
import os
import sys
import time
import copy
import json
import subprocess
from pathlib import Path
from typing import Dict, Any, List

# Ensure notebook can import project modules when running from notebooks/
WORKSPACE_ROOT = Path.cwd().resolve().parent
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import set_key, dotenv_values
from huggingface_hub import HfApi
from IPython.display import display
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

from src.finetuner.data_loader import GLUEDataLoader
from src.utils.metrics import UnifiedEvaluator

print("Loaded notebook experiment dependencies.")
print("Workspace root:", WORKSPACE_ROOT)
print("Main execution route: uv run python -c 'import main; main.main()'")

Loaded notebook experiment dependencies.
Workspace root: C:\tutoriales\fineTunning
Main execution route: uv run python -c 'import main; main.main()'


In [5]:
#GLUE_TASKS = ["mrpc", "sst2", "qnli"]
GLUE_TASKS = ["mrpc"]
OUTPUT_ROOT = (WORKSPACE_ROOT / "output").resolve()
LOG_ROOT = (WORKSPACE_ROOT / "logs").resolve()
ENV_PATH = (WORKSPACE_ROOT / ".env").resolve()

SUITE_OUTPUT_ROOTS = {
    "LoRA Only": (OUTPUT_ROOT / "lora_only").resolve(),
    "Laplace-LoRA Only": (OUTPUT_ROOT / "laplace_lora").resolve(),
    "Causal-LoRA Only": (OUTPUT_ROOT / "causal_lora").resolve(),
}

BASE_ENV = {
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "MODEL_ID": "bert-base-uncased",
    "BATCH_SIZE": "8",
    "EPOCHS": "1",
    "LEARNING_RATE": "2e-5",
    "LOGGING_DIR": str(LOG_ROOT),
    "LORA_RANK": "8",
    "LORA_ALPHA": "16",
    "LORA_DROPOUT": "0.1",
    "APPLY_INTERVAL": "10",
    "CAUSAL_SOFTMAX_TEMP_INITIAL": "2.0",
    "CAUSAL_SOFTMAX_TEMP_FINAL": "0.5",
    "CAUSAL_TEMP_SCHEDULE": "linear_decay",
    "MAX_RAM_THRESHOLD_GB": "9.0",
    "DEFAULT_QUANTIZATION": "auto",
    "ENABLE_PG_POS": "false",
    "KFAC_CORRELATION": "false",
    "RESUME_POLICY": "strict",
    "ENABLE_INTERVENTIONAL_WEIGHTS": "false",
}

EXPERIMENT_SUITES = {
    "LoRA Only": {
        "EXPERIMENT_TYPE": "lora",
        "EXECUTE_CAUSAL_ENGINE": "false",
        "EXECUTE_LAPLACE": "false",
        "CAUSAL_SAMPLER_MODE": "gradient",
        "OUTPUT_ROOT": str(SUITE_OUTPUT_ROOTS["LoRA Only"]),
    },
    "Laplace-LoRA Only": {
        "EXPERIMENT_TYPE": "laplace_lora",
        "EXECUTE_CAUSAL_ENGINE": "false",
        "EXECUTE_LAPLACE": "true",
        "CAUSAL_SAMPLER_MODE": "gradient",
        "OUTPUT_ROOT": str(SUITE_OUTPUT_ROOTS["Laplace-LoRA Only"]),
    },
    "Causal-LoRA Only": {
        "EXPERIMENT_TYPE": "causal_lora",
        "EXECUTE_CAUSAL_ENGINE": "true",
        "EXECUTE_LAPLACE": "false",
        "CAUSAL_SAMPLER_MODE": "mixture_of_gaussians",
        "ENABLE_PG_POS": "true",
        "KFAC_CORRELATION": "true",
        "ENABLE_INTERVENTIONAL_WEIGHTS": "true",
        "OUTPUT_ROOT": str(SUITE_OUTPUT_ROOTS["Causal-LoRA Only"]),
    },
}

METRIC_COLUMNS = [
    "accuracy",
    "f1",
    "nll",
    "ece",
    "posterior_mean_nll",
    "posterior_variance_mean",
    "epistemic_uncertainty_mean",
]

print("GLUE tasks:", GLUE_TASKS)
print("Experiment suites:", list(EXPERIMENT_SUITES.keys()))
print("Suite output roots:")
for _suite_name, _root in SUITE_OUTPUT_ROOTS.items():
    print(f"  {_suite_name}: {_root}")

GLUE tasks: ['mrpc']
Experiment suites: ['LoRA Only', 'Laplace-LoRA Only', 'Causal-LoRA Only']
Suite output roots:
  LoRA Only: C:\tutoriales\fineTunning\output\lora_only
  Laplace-LoRA Only: C:\tutoriales\fineTunning\output\laplace_lora
  Causal-LoRA Only: C:\tutoriales\fineTunning\output\causal_lora


In [ ]:
def _ensure_env_file() -> None:
    if not ENV_PATH.exists():
        ENV_PATH.write_text("", encoding="utf-8")


def _validate_hf_token(token: str) -> str:
    token = token.strip()
    if not token:
        raise ValueError("HF_TOKEN is missing. Set a valid token in environment or .env before running experiments.")
    try:
        HfApi().whoami(token=token)
    except Exception as exc:
        raise ValueError(
            "HF_TOKEN is invalid. Update HF_TOKEN in environment or .env with a valid Hugging Face token."
        ) from exc
    return token


def _resolve_hf_token() -> str:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return _validate_hf_token(token)

    env_data = dotenv_values(str(ENV_PATH), encoding="utf-8") if ENV_PATH.exists() else {}
    token = str(env_data.get("HF_TOKEN", "")).strip()
    return _validate_hf_token(token)


def _prepare_trial_state(seed: int = 42) -> None:
    import gc
    import random

    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.empty_cache()
    gc.collect()


def _cleanup_trial_state() -> None:
    import gc

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _build_trial_env(env_updates: Dict[str, str]) -> Dict[str, str]:
    child_env = os.environ.copy()
    for key, value in env_updates.items():
        child_env[key] = str(value)
    return child_env


def _find_latest_model_dir(run_root: Path) -> Path:
    checkpoints = [p for p in run_root.glob("checkpoint-*") if p.is_dir()]
    if not checkpoints:
        return run_root

    def _step_num(path_obj: Path) -> int:
        try:
            return int(path_obj.name.split("-")[1])
        except Exception:
            return -1

    checkpoints.sort(key=_step_num)
    return checkpoints[-1]


def _read_checkpoint_epoch(checkpoint_dir: Path) -> Any:
    state_path = checkpoint_dir / "trainer_state.json"
    if not state_path.exists():
        return None
    try:
        payload = json.loads(state_path.read_text(encoding="utf-8"))
        if payload.get("epoch") is not None:
            return payload.get("epoch")
        history = payload.get("log_history", [])
        epochs = [entry.get("epoch") for entry in history if isinstance(entry, dict) and entry.get("epoch") is not None]
        return epochs[-1] if epochs else None
    except Exception:
        return None


def _run_main_with_env(env_updates: Dict[str, str]) -> Dict[str, Any]:
    _prepare_trial_state()
    child_env = _build_trial_env(env_updates)
    cmd = [
        "uv",
        "run",
        "python",
        "-c",
        "import json, main; print(json.dumps(main.main()))",
    ]
    try:
        proc = subprocess.run(
            cmd,
            cwd=str(WORKSPACE_ROOT),
            env=child_env,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            check=False,
        )
        if proc.returncode != 0:
            stderr = (proc.stderr or "").strip()
            stdout = (proc.stdout or "").strip()
            raise RuntimeError(f"main.py execution failed (code={proc.returncode}). stdout={stdout} stderr={stderr}")

        lines = [line for line in (proc.stdout or "").splitlines() if line.strip()]
        if not lines:
            return {"status": "failure", "error_type": "EmptyOutput", "error_message": "No JSON output from main.py"}

        try:
            return json.loads(lines[-1])
        except Exception as exc:
            raise RuntimeError(f"Could not parse main.py JSON output. Last line: {lines[-1]!r}") from exc
    finally:
        _cleanup_trial_state()


def _fresh_retry_env(env_updates: Dict[str, str], task_name: str) -> Dict[str, str]:
    retry_updates = copy.deepcopy(env_updates)
    retry_base = Path(env_updates["OUTPUT_DIR"]).resolve().parent / f"{Path(env_updates['OUTPUT_DIR']).name}_retry_{int(time.time())}"
    retry_updates["OUTPUT_DIR"] = str(retry_base)
    print(f"Retrying with fresh training root due to source-side checkpoint rejection: {retry_base}_{task_name}")
    return retry_updates


def _load_trained_model(model_dir: Path, base_model_id: str):
    model_dir = model_dir.resolve()
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory does not exist: {model_dir}")

    adapter_config = model_dir / "adapter_config.json"
    config_json = model_dir / "config.json"

    if adapter_config.exists():
        base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id)
        model = PeftModel.from_pretrained(base_model, str(model_dir))
        return model

    if config_json.exists():
        return AutoModelForSequenceClassification.from_pretrained(str(model_dir))

    raise FileNotFoundError(
        f"No compatible model artifacts found in {model_dir}. Expected adapter_config.json or config.json."
    )


def _evaluate_saved_model(task_name: str, model_id: str, model_dir: Path, eval_output_dir: Path) -> Dict[str, Any]:
    loader = GLUEDataLoader(model_id, task_name)
    _, eval_ds, _ = loader.get_datasets(train_split="train")
    evaluator = UnifiedEvaluator(task_name)

    _prepare_trial_state()
    model = None
    eval_trainer = None
    pred = None
    try:
        model = _load_trained_model(model_dir, model_id)
        model.eval()

        eval_args = TrainingArguments(
            output_dir=str(eval_output_dir.resolve()),
            per_device_eval_batch_size=8,
            report_to=[],
            logging_strategy="no",
        )
        eval_trainer = Trainer(model=model, args=eval_args)
        pred = eval_trainer.predict(eval_ds)

        logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
        return evaluator.compute_all(torch.tensor(logits), torch.tensor(pred.label_ids))
    finally:
        del pred
        del eval_trainer
        del model
        _cleanup_trial_state()


def _evaluate_run_by_epoch(task_name: str, model_id: str, run_dir: Path, suite_name: str) -> pd.DataFrame:
    checkpoints = [p for p in run_dir.glob("checkpoint-*") if p.is_dir()]

    def _step_num(path_obj: Path) -> int:
        try:
            return int(path_obj.name.split("-")[1])
        except Exception:
            return -1

    checkpoints.sort(key=_step_num)

    rows: List[Dict[str, Any]] = []
    eval_root = run_dir / "_eval_tmp" / suite_name.lower().replace("-", "_").replace(" ", "_")
    if not checkpoints:
        metrics = _evaluate_saved_model(task_name, model_id, run_dir, eval_root / "final")
        rows.append({
            "epoch_index": 1,
            "epoch": None,
            "checkpoint": run_dir.name,
            "accuracy": metrics.get("accuracy"),
            "f1": metrics.get("f1"),
            "nll": metrics.get("nll"),
            "ece": metrics.get("ece"),
            "posterior_mean_nll": metrics.get("posterior_mean_nll"),
            "posterior_variance_mean": metrics.get("posterior_variance_mean"),
            "epistemic_uncertainty_mean": metrics.get("epistemic_uncertainty_mean"),
        })
        return pd.DataFrame(rows)

    for idx, checkpoint_dir in enumerate(checkpoints, start=1):
        metrics = _evaluate_saved_model(
            task_name,
            model_id,
            checkpoint_dir,
            eval_root / checkpoint_dir.name,
        )
        rows.append({
            "epoch_index": idx,
            "epoch": _read_checkpoint_epoch(checkpoint_dir),
            "checkpoint": checkpoint_dir.name,
            "accuracy": metrics.get("accuracy"),
            "f1": metrics.get("f1"),
            "nll": metrics.get("nll"),
            "ece": metrics.get("ece"),
            "posterior_mean_nll": metrics.get("posterior_mean_nll"),
            "posterior_variance_mean": metrics.get("posterior_variance_mean"),
            "epistemic_uncertainty_mean": metrics.get("epistemic_uncertainty_mean"),
        })

    return pd.DataFrame(rows)


def _print_epoch_metrics(epoch_df: pd.DataFrame, suite_name: str, task_name: str) -> None:
    print(f"\n{'='*120}")
    print(f"Per-Epoch Metrics: {suite_name} | Task: {task_name}")
    print(f"{'='*120}")

    display_cols = ["epoch_index", "epoch", "checkpoint"] + METRIC_COLUMNS
    available_cols = [col for col in display_cols if col in epoch_df.columns]
    display_df = epoch_df[available_cols].copy()

    for col in METRIC_COLUMNS:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{float(x):.4f}" if pd.notna(x) else "N/A")

    print()
    display(display_df)
    print()


def _print_run_details(run_result: Dict[str, Any]) -> None:
    print(f"Run details | suite={run_result['suite']} task={run_result['task']} status={run_result['status']}")
    detail_cols = [
        'suite', 'task', 'status', 'output_dir', 'model_dir',
        'runtime_seconds', 'resume_checkpoint', 'causal_budget_snapshot', 'causal_budget_utilization'
    ]
    details = {
        key: run_result.get(key)
        for key in detail_cols
    }
    details_df = pd.DataFrame([details])
    display(details_df)
    print()


def run_suite_task(suite_name: str, task_name: str) -> Dict[str, Any]:
    _prepare_trial_state()
    try:
        suite_cfg = copy.deepcopy(EXPERIMENT_SUITES[suite_name])
        run_output_base = Path(suite_cfg.pop("OUTPUT_ROOT")).resolve()
        log_output_dir = (LOG_ROOT / suite_cfg["EXPERIMENT_TYPE"] / task_name).resolve()
        run_dir = Path(f"{run_output_base}_{task_name}")

        env_updates = copy.deepcopy(BASE_ENV)
        env_updates.update(suite_cfg)
        env_updates.update({
            "HF_TOKEN": _resolve_hf_token(),
            "TASK_NAME": task_name,
            "OUTPUT_DIR": str(run_output_base),
            "LOGGING_DIR": str(log_output_dir),
            "RESUME_POLICY": BASE_ENV["RESUME_POLICY"],
        })

        started = time.time()
        main_result = _run_main_with_env(env_updates)

        if main_result.get("status") == "failure" and "resume_no_valid_checkpoint" in json.dumps(main_result):
            env_updates = _fresh_retry_env(env_updates, task_name)
            main_result = _run_main_with_env(env_updates)

        run_dir = Path(f"{env_updates['OUTPUT_DIR']}_{task_name}")
        if not run_dir.exists():
            raise RuntimeError(
                f"Expected training output directory was not created: {run_dir}. main.py returned {main_result!r}"
            )

        epoch_df = _evaluate_run_by_epoch(task_name, env_updates["MODEL_ID"], run_dir, suite_name)
        _print_epoch_metrics(epoch_df, suite_name, task_name)

        final_metrics = epoch_df.iloc[-1].to_dict()
        final_model_dir = run_dir / str(final_metrics.get("checkpoint")) if final_metrics.get("checkpoint") else _find_latest_model_dir(run_dir)

        diagnostics = (main_result or {}).get("diagnostics") or {}
        run_result = {
            "suite": suite_name,
            "task": task_name,
            "status": (main_result or {}).get("status", "unknown"),
            "output_dir": str(run_dir),
            "model_dir": str(final_model_dir),
            "resume_checkpoint": (main_result or {}).get("resume_checkpoint"),
            "causal_budget_snapshot": diagnostics.get("budget_snapshot"),
            "causal_budget_utilization": diagnostics.get("budget_utilization"),
            "runtime_seconds": time.time() - started,
            "accuracy": final_metrics.get("accuracy"),
            "f1": final_metrics.get("f1"),
            "nll": final_metrics.get("nll"),
            "ece": final_metrics.get("ece"),
            "posterior_mean_nll": final_metrics.get("posterior_mean_nll"),
            "posterior_variance_mean": final_metrics.get("posterior_variance_mean"),
            "epistemic_uncertainty_mean": final_metrics.get("epistemic_uncertainty_mean"),
        }
        _print_run_details(run_result)
        return run_result
    finally:
        _cleanup_trial_state()


def run_experiment_suite(suite_name: str, tasks: List[str]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    print(f"\n{'#'*120}")
    print(f"# Starting Experiment Suite: {suite_name}")
    print(f"{'#'*120}\n")

    for task_name in tasks:
        print(f"Running {suite_name} | task={task_name}")
        rows.append(run_suite_task(suite_name, task_name))

    result_df = pd.DataFrame(rows)
    print(f"\n{'='*120}")
    print(f"Summary for {suite_name} Suite")
    print(f"{'='*120}")
    display(result_df[["task", "status", *METRIC_COLUMNS, "runtime_seconds"]])

    return result_df


print("Helpers ready. HF token source check passed:", bool(_resolve_hf_token()))

Helpers ready. HF token source check passed: True


## Experiment 1: LoRA Only (3 GLUE tasks)

This run executes `main.py` with LoRA-only settings for `mrpc`, `sst2`, and `qnli` and computes the same evaluation metrics used by all suites.

In [11]:
lora_df = run_experiment_suite("LoRA Only", GLUE_TASKS)
lora_df

Running LoRA Only | task=mrpc
2026-04-09 17:52:08,099 - FineTuningApp - INFO - Multiprocessing start method configured to 'spawn'.
2026-04-09 17:52:08,102 - FineTuningApp - INFO - Multiprocessing runtime | requested=spawn current=spawn env=local
2026-04-09 17:52:08,105 - FineTuningApp - INFO - Initializing Fine-tuning Orchestrator...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-09 17:52:13,014 - FineTuningApp - INFO - Applying LoRA (r=8, alpha=16) to base model...
2026-04-09 17:52:13,202 - FineTuningApp - INFO - LoRA applied successfully. Trainable parameters: 296,450
2026-04-09 17:52:13,263 - FineTuningApp - INFO - Resume disabled by policy=false


c:\tutoriales\fineTunning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': '1731', 'train_samples_per_second': '2.119', 'train_steps_per_second': '0.265', 'train_loss': '0.6431', 'epoch': '1'}
2026-04-09 18:21:04,961 - FineTuningApp - INFO - Causal engine disabled; completed standard LoRA training.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\tutoriales\fineTunning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Per-epoch metrics | suite=LoRA Only | task=mrpc


,epoch_index,epoch,checkpoint,accuracy,f1,nll,ece,posterior_mean_nll,posterior_variance_mean,epistemic_uncertainty_mean
0,1,1.0,checkpoint-459,0.683824,0.406114,0.624429,0.018759,None,None,None


,suite,task,output_dir,model_dir,resume_checkpoint,causal_budget_snapshot,causal_budget_utilization,runtime_seconds,accuracy,f1,nll,ece,posterior_mean_nll,posterior_variance_mean,epistemic_uncertainty_mean
0,LoRA Only,mrpc,C:\tutoriales\fineTunning\output\lora_only_mrpc,C:\tutoriales\fineTunning\output\lora_only_mrp...,None,None,None,1787.594974,0.683824,0.406114,0.624429,0.018759,None,None,None


## Experiment 2: Laplace-LoRA Only (3 GLUE tasks)

In [14]:
laplace_df = run_experiment_suite("Laplace-LoRA Only", GLUE_TASKS)
laplace_df

Running Laplace-LoRA Only | task=mrpc
2026-04-09 18:22:41,916 - FineTuningApp - INFO - Multiprocessing start method configured to 'spawn'.
2026-04-09 18:22:41,921 - FineTuningApp - INFO - Multiprocessing runtime | requested=spawn current=spawn env=local
2026-04-09 18:22:41,924 - FineTuningApp - INFO - Initializing Fine-tuning Orchestrator...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-09 18:22:45,295 - FineTuningApp - INFO - Applying LoRA (r=8, alpha=16) to base model...
2026-04-09 18:22:45,392 - FineTuningApp - INFO - LoRA applied successfully. Trainable parameters: 296,450
2026-04-09 18:22:45,423 - FineTuningApp - INFO - Resume disabled by policy=false


c:\tutoriales\fineTunning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': '1.805e+05', 'train_samples_per_second': '0.02', 'train_steps_per_second': '0.003', 'train_loss': '0.6391', 'epoch': '1'}
2026-04-11 20:31:18,154 - FineTuningApp - INFO - Causal engine disabled; completed standard LoRA training.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\tutoriales\fineTunning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Per-epoch metrics | suite=Laplace-LoRA Only | task=mrpc


,epoch_index,epoch,checkpoint,accuracy,f1,nll,ece,posterior_mean_nll,posterior_variance_mean,epistemic_uncertainty_mean
0,1,1.0,checkpoint-459,0.683824,0.406114,0.622013,0.043165,None,None,None


,suite,task,output_dir,model_dir,resume_checkpoint,causal_budget_snapshot,causal_budget_utilization,runtime_seconds,accuracy,f1,nll,ece,posterior_mean_nll,posterior_variance_mean,epistemic_uncertainty_mean
0,Laplace-LoRA Only,mrpc,C:\tutoriales\fineTunning\output\laplace_lora_...,C:\tutoriales\fineTunning\output\laplace_lora_...,None,None,None,180590.938313,0.683824,0.406114,0.622013,0.043165,None,None,None


## Experiment 3: Causal-LoRA Only (3 GLUE tasks)

Causal flow enforces mandatory prepare() before training and uses strict auto-resume.
The result row now includes causal budget snapshot and utilization from runtime diagnostics.

In [7]:
causal_df = run_experiment_suite("Causal-LoRA Only", GLUE_TASKS)
causal_df


########################################################################################################################
# Starting Experiment Suite: Causal-LoRA Only
########################################################################################################################

Running Causal-LoRA Only | task=mrpc


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Python314\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "C:\Python314\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Python314\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 694: character maps to <undefined>


RuntimeError: main.py execution failed (code=1). stdout=2026-04-12 12:41:13,861 - FineTuningApp - INFO - Multiprocessing start method configured to 'spawn'.
2026-04-12 12:41:13,861 - FineTuningApp - INFO - Multiprocessing runtime | requested=spawn current=spawn env=local
2026-04-12 12:41:13,861 - FineTuningApp - INFO - Initializing Fine-tuning Orchestrator...
2026-04-12 12:41:15,499 - FineTuningApp - INFO - Downloading GLUE task: mrpc
2026-04-12 12:41:20,655 - FineTuningApp - INFO - Applying LoRA (r=8, alpha=16) to base model...
2026-04-12 12:41:22,122 - FineTuningApp - INFO - LoRA applied successfully. Trainable parameters: 296,450
2026-04-12 12:41:22,137 - FineTuningApp - INFO - Strict resume found no valid checkpoint; starting from clean state.
2026-04-12 12:41:22,138 - FineTuningApp - INFO - Using NIEBudgetAllocationStrategy (Ï„_init=2.0000, Ï„_final=0.5000, anneal=True)
2026-04-12 12:41:22,162 - FineTuningApp - INFO - CausalMonteCLoRAEngine initialized with threshold=0.1, budget=1

## Unified Comparison (same metrics across all experiments)

This table combines all runs and compares the complete contract metrics (core + Laplace keys) for every suite-task pair.

In [ ]:
comparison_df = pd.concat([lora_df, laplace_df, causal_df], ignore_index=True)
comparison_df = comparison_df[["suite", "task", "status", *METRIC_COLUMNS, "runtime_seconds", "resume_checkpoint", "causal_budget_snapshot", "causal_budget_utilization", "output_dir", "model_dir"]]

print("\n" + "="*120)
print("UNIFIED MULTI-SUITE COMPARISON: Final Metrics Across All Experiments")
print("="*120 + "\n")
display(comparison_df.sort_values(["suite", "task"]).reset_index(drop=True))

print("\n" + "="*120)
print("SUMMARY STATISTICS BY SUITE")
print("="*120 + "\n")
for suite_name in ["LoRA Only", "Laplace-LoRA Only", "Causal-LoRA Only"]:
    suite_data = comparison_df[comparison_df["suite"] == suite_name]
    if len(suite_data) > 0:
        print(f"\n{suite_name}:")
        print(f"  Tasks completed: {len(suite_data)}")
        for metric in ["accuracy", "f1", "nll", "ece"]:
            if metric in suite_data.columns:
                values = suite_data[metric].dropna()
                if len(values) > 0:
                    print(f"  {metric:20s}: mean={values.mean():.4f}, final={suite_data[metric].iloc[-1]:.4f}")
        runtime_total = suite_data["runtime_seconds"].sum()
        print(f"  Total runtime: {runtime_total:.1f} seconds\n")

plot_df = comparison_df.sort_values(["suite", "task"]).reset_index(drop=True).copy()
plot_df["run_label"] = plot_df["suite"] + " | " + plot_df["task"]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].bar(plot_df["run_label"], plot_df["accuracy"], color=["#4C78A8", "#F58518", "#54A24B"])
axes[0].set_title("Accuracy Comparison")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(plot_df["run_label"], plot_df["nll"], color=["#4C78A8", "#F58518", "#54A24B"])
axes[1].set_title("Loss Comparison (NLL)")
axes[1].set_ylabel("NLL (lower is better)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print("="*120)